# Chapitre 16 - Cours 1 : boosting et chemins de régularisation

Ce premier notebook couvre la section 16.2 du livre. Nous y montrons que le
**boosting avec rétrécissement** (« shrinkage ») se comporte comme un **lasso** posé
dans l'espace gigantesque de toutes les fonctions de base possibles (par exemple
tous les arbres), puis nous en tirons trois leçons :

1. la régression *forward stagewise* incrémentale (algorithme 16.1) suit
   approximativement le chemin du lasso (figure 16.1) ;
2. le principe du **« pari sur la parcimonie »** (*bet on sparsity*) explique
   pourquoi une pénalité $L_1$ est préférable en grande dimension ;
3. les chemins de régularisation du boosting sont « bien élevés » (monotones,
   figures 16.3–16.4) et convergent vers une solution de **marge $L_1$ maximale**
   (figure 16.5).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch16_ensembles import (standardize, forward_stagewise, lasso_path_cd,
                                ridge_path_df, block_correlated_data,
                                l1_arc_length, adaboost_shrunk)

## 1. Qu'est-ce qu'un ensemble ?

L'**apprentissage ensembliste** construit un prédicteur en combinant de nombreux
modèles simples, appelés *apprenants de base* ou *apprenants faibles*. Nous en avons
déjà rencontré plusieurs :

- le **bagging** (section 8.7) et les **forêts aléatoires** (chapitre 15) : un
  comité d'arbres vote ;
- le **boosting** (chapitre 10) : le comité est construit séquentiellement et les
  votes sont pondérés ;
- le **stacking** (section 8.8) ;
- plus généralement toute méthode à *dictionnaire* (splines de régression, etc.),
  où les fonctions de base jouent le rôle des apprenants faibles.

On peut décomposer la démarche en deux tâches : (i) produire une population
d'apprenants de base à partir des données ; (ii) les combiner en un prédicteur final.

> **Remarque (codes correcteurs d'erreurs).** Un ancêtre des ensembles est la méthode
> ECOC pour la classification à $K$ classes : chaque classe $k$ reçoit un mot de code
> binaire $(C_{k1},\dots,C_{kL})$ (une ligne de la matrice de codage, tableau 16.1 du
> livre) ; on apprend un classifieur à deux classes par colonne $\ell$, qui estime
> $\hat p_\ell(x)$, la probabilité que le bit $\ell$ vaille 1 ; puis on affecte $x$ à la
> classe minimisant $\delta_k(x)=\sum_{\ell=1}^L |C_{k\ell}-\hat p_\ell(x)|$. Les bits
> redondants permettent de « corriger » quelques erreurs des classifieurs ; en pratique
> le gain vient surtout d'une réduction de variance, comme pour le bagging.